# Train a Codenames spymaster with GRPO

This notebook trains **Qwen2.5-1.5B-Instruct** (with LoRA) to be a better Codenames spymaster, using
reinforcement learning (GRPO). The guesser is **Llama 3.3 70B via Inworld**, the same guesser as in the baseline.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (free).
2. Click the 🔑 icon on the left (Secrets) → *Add new secret*: name `INWORLD_API_KEY`, value = your key. Turn on *Notebook access*.
3. *Runtime → Run all*. Google Drive asks for permission: all results are saved there, so nothing is lost if Colab disconnects.

**Steps**
1. Setup: code from GitHub, packages, Drive, API key
2. Load the model and measure it before training
3. Train (about 1–1.5 hours on a T4, about $0.50–1 for guesser calls)
4. Measure after training and save the LoRA adapter to Drive
5. Export to GGUF (Q4_K_M, the same compression as Ollama) for the real comparison on a laptop

Did Colab disconnect? Just *Run all* again: training continues from the last checkpoint on Drive.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/casruger/code-names-trained-llm"

import os
if not os.path.exists("/content/code-names-trained-llm"):
    !git clone -q {REPO_URL} /content/code-names-trained-llm
else:
    !git -C /content/code-names-trained-llm pull -q
%cd /content/code-names-trained-llm
!pip install -q trl peft datasets openai python-dotenv
# Colab ships an old torchao that the new peft refuses to import; we don't need it.
!pip uninstall -y -q torchao

In [ ]:
# ---- settings --------------------------------------------------------------------
RUN_NAME        = "grpo-v1"        # new name = new training run (own folder on Drive)
MODEL_ID        = "Qwen/Qwen2.5-1.5B-Instruct"   # same model as qwen2.5:1.5b in Ollama
GUESSER_MODEL   = "deepinfra/meta-llama/Llama-3.3-70B-Instruct-Turbo"  # do NOT change (must match the baseline)

N_TRAIN_BOARDS  = 3000     # training boards (seeds 0..2999; evaluation boards start at 1,000,000)
MAX_STEPS       = 300      # optimizer steps; each step = 4 boards x 8 clues
NUM_GENERATIONS = 8        # how many clues per board GRPO compares with each other
BATCH_SIZE      = 8        # clues per forward pass (must be divisible by NUM_GENERATIONS)
GRAD_ACCUM      = 4        # -> 8 x 4 = 32 clues = 4 boards per step
LEARNING_RATE   = 1e-5
LORA_R          = 16
BETA            = 0.04     # KL penalty: keeps the model close to the original (0 = free)
TEMPERATURE     = 1.0      # sampling during training: we need variety to compare clues
EVAL_BOARDS     = 50       # the same 50 evaluation boards as baseline.py
# ---------------------------------------------------------------------------------

from google.colab import drive, userdata
drive.mount("/content/drive")
OUT = f"/content/drive/MyDrive/codenames-rl/{RUN_NAME}"
os.makedirs(OUT, exist_ok=True)
os.environ["INWORLD_API_KEY"] = userdata.get("INWORLD_API_KEY")
print("Everything is saved in", OUT)

In [ ]:
# Test the guesser (1 API call)
from codenames import generate_board
from codenames.players import LLMClient, LLMGuesser

guesser = LLMGuesser(LLMClient(GUESSER_MODEL, "inworld", temperature=0.0))
b = generate_board(seed=42)
team = b.hidden_by_role("team")[:2]
print("team words:", team)
print("guesser says:", guesser.guess(b, "test", 2), "| raw:", guesser.last_raw)

## 2. Load the model and measure it before training

This uses the same boards, prompts and scoring as `baseline.py`, but with the model in Hugging Face
format (16-bit). The score can be a bit different from the Ollama baseline (−2.16, 4-bit). The official
comparison is done later on a laptop with Ollama. Here it is a quick check for this notebook.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# bf16 only on GPUs with compute capability >= 8 (A100, L4). On a T4 (7.5)
# torch.cuda.is_bf16_supported() says "True" (emulation), but training then crashes -> use fp16.
USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
if not USE_BF16:
    try:  # TRL's fast fp32 matmul only works on newer GPUs; use the safe version
        import trl.kernels.chunked_logprob as _clp
        _clp._MM_OUT_DTYPE = False
    except ImportError:
        pass
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU!", "| dtype:", DTYPE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map="cuda")

In [ ]:
import json
from codenames.training import evaluate_hf, CachedGuesser

eval_guesser = CachedGuesser(guesser)
before, before_rows = evaluate_hf(model, tokenizer, eval_guesser, n=EVAL_BOARDS)
json.dump({"summary": before, "rows": before_rows}, open(f"{OUT}/eval_before.json", "w"))
for k, v in before.items(): print(f"{k:<20}{v:8.2f}")

## 3. Train with GRPO

Each step: the model writes 8 clues per board → the guesser guesses → `score_clue_text` gives the reward →
clues that are better than the group average become more likely.

Watch these numbers:
- **`reward`**: average reward of the clues. It should slowly go up.
- **`frac_reward_zero_std`**: share of boards where all 8 clues got the same score. Then there is nothing
  to compare, so nothing to learn. If this stays close to 1, the model does not learn.
- **`kl`**: how far the model moved away from the original. A slow increase is normal.

The training loss stays close to 0. That is normal for GRPO and does not show progress.
Every sample (clue, guesses, reward) is saved in `reward_log.jsonl` on Drive.

In [ ]:
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer
from codenames.training import build_dataset, CodenamesReward

train_dataset = build_dataset(N_TRAIN_BOARDS, seed=0)
reward_fn = CodenamesReward(guesser, max_workers=16, log_path=f"{OUT}/reward_log.jsonl")

lora = LoraConfig(
    r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
config = GRPOConfig(
    output_dir=f"{OUT}/checkpoints",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_generations=NUM_GENERATIONS,
    max_completion_length=32,          # 'CLUE: x / NUMBER: n' is short; longer = wasted time
    temperature=TEMPERATURE,
    beta=BETA,
    max_steps=MAX_STEPS,
    logging_steps=5,
    save_steps=25,                     # checkpoint on Drive -> can continue after a disconnect
    save_total_limit=2,
    bf16=USE_BF16, fp16=not USE_BF16,
    gradient_checkpointing=True,
    report_to="none",
    seed=0,
)
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],
    args=config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=lora,
)
trainer.model.print_trainable_parameters()

In [ ]:
import glob
resume = bool(glob.glob(f"{OUT}/checkpoints/checkpoint-*"))
print("Continue from checkpoint" if resume else "New training run")
trainer.train(resume_from_checkpoint=resume)
print(f"guesser cache: {reward_fn.guesser.hits} hits, {reward_fn.guesser.misses} API calls")

In [ ]:
# How did it go? Reward per step + what kind of clues the model gives now.
import collections
import matplotlib.pyplot as plt
import pandas as pd
from codenames.training import read_reward_log

hist = pd.DataFrame([h for h in trainer.state.log_history if "reward" in h])
log = pd.DataFrame(read_reward_log(f"{OUT}/reward_log.jsonl"))
log["valid"] = log["error"].isna()
w = max(1, len(log) // 40)

fig, ax = plt.subplots(1, 4, figsize=(20, 4))
ax[0].plot(hist["step"], hist["reward"]); ax[0].set_title("mean reward per step")
ax[1].plot(log["valid"].rolling(w * 32).mean().values); ax[1].set_title("share of valid clues")
ax[2].plot(log["n_correct"].rolling(w * 32).mean().values); ax[2].set_title("mean correct guesses per clue")
ax[3].plot(hist["step"], hist["frac_reward_zero_std"]); ax[3].set_title("boards without learning signal\n(all 8 clues scored the same)")
for a in ax: a.grid(alpha=.3)
plt.show()

last = log.tail(min(1000, len(log)))
print("Most common clues in the last 1000 samples (watch for reward hacking!):")
print(collections.Counter(last.loc[last.valid, "clue"]).most_common(15))
print("\nErrors:", collections.Counter(e.split("'")[0] for e in last.error.dropna()).most_common(5))

## 4. Measure after training and save

In [ ]:
after, after_rows = evaluate_hf(trainer.model, tokenizer, eval_guesser, n=EVAL_BOARDS)
json.dump({"summary": after, "rows": after_rows}, open(f"{OUT}/eval_after.json", "w"))
print(f"{'metric':<20}{'before':>8}{'after':>8}")
for k in before: print(f"{k:<20}{before[k]:8.2f}{after[k]:8.2f}")
print("\nExamples after training:")
for r in after_rows[:8]: print(f"  {r['clue']!s:>12} {r['number']!s:>2} | {r['reward']:+.1f} | {r['error'] or r['outcomes']}")

In [ ]:
# The LoRA adapter (~20-40 MB) = what the model has learned
ADAPTER_DIR = f"{OUT}/lora_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
json.dump({k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str))},
          open(f"{OUT}/config.json", "w"), indent=2)
print("Adapter saved in", ADAPTER_DIR)

## 5. Export to GGUF for Ollama

The adapter and the base model are merged and compressed to **Q4_K_M**. This is the same compression as
`qwen2.5:1.5b` in Ollama, so the comparison with the baseline is fair. This takes about 5–10 minutes.

In [ ]:
merged = trainer.model.merge_and_unload()
merged.save_pretrained("/content/merged", safe_serialization=True)
tokenizer.save_pretrained("/content/merged")

!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q sentencepiece gguf
!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged --outfile /content/spymaster-f16.gguf --outtype f16
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4 > /dev/null
!/content/llama.cpp/build/bin/llama-quantize /content/spymaster-f16.gguf /content/codenames-spymaster-q4_k_m.gguf Q4_K_M
!cp /content/codenames-spymaster-q4_k_m.gguf "{OUT}/"
!ls -lh "{OUT}"

## 6. On your laptop: import into Ollama and compare

1. Download `codenames-spymaster-q4_k_m.gguf` from Google Drive (`codenames-rl/grpo-v1/`) into the
   `trained_model/` folder of the project.
2. In **PowerShell** (not cmd), copy the settings of the original model and only swap the weights:
   ```powershell
   cd trained_model
   ollama show qwen2.5:1.5b --modelfile > Modelfile
   (Get-Content Modelfile) -replace '^FROM .*', 'FROM ./codenames-spymaster-q4_k_m.gguf' | Set-Content Modelfile
   ollama create codenames-spymaster -f Modelfile
   cd ..
   ```
   Check with `ollama list` that `codenames-spymaster` has a different ID than `qwen2.5:1.5b`.
3. Run the same evaluation as the baseline, from the project folder:
   ```powershell
   python baseline.py --model codenames-spymaster --guesser-provider inworld --guesser-model deepinfra/meta-llama/Llama-3.3-70B-Instruct-Turbo --runs 5
   ```
4. Compare both runs and make the chart:
   ```powershell
   python compare.py eval_results/qwen2.5_1.5b__vs__deepinfra_meta-llama_Llama-3.3-70B-Instruct-Turbo eval_results/codenames-spymaster__vs__deepinfra_meta-llama_Llama-3.3-70B-Instruct-Turbo
   ```